# Curs 2 — Ecosistemul de modele

Scopul acestui notebook: testăm **2-3 modele diferite** pe același input și alegem modelul potrivit pentru proiect.

Vom folosi:
1. **Gemini** — providerul principal, prin cheia obținută din Google AI Studio.
2. **OpenRouter** — provider alternativ, util pentru comparație și backup când Gemini are limite de quota.
## OpenRouter — de unde luăm cheia
1. Intră pe https://openrouter.ai/
2. Creează cont sau autentifică-te.
3. Mergi la **Keys**.
4. Creează un nou API key.
5. Copiază cheia în fișierul `.env`:
```env
OPENROUTER_API_KEY=pune-cheia-ta-aici
---

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json

## 1. Configurare — mai multe modele

In [2]:
MODELE = [
    ("gemini", "gemini-2.5-flash-lite", "Gemini 2.5 Flash Lite"),
    ("gemini", "gemini-2.5-flash", "Gemini 2.5 Flash"),
    ("openrouter", "openrouter/free", "OpenRouter Free"),
]

print("Modele pregătite:", [nume for _, _, nume in MODELE])

Modele pregătite: ['Gemini 2.5 Flash Lite', 'Gemini 2.5 Flash', 'OpenRouter Free']


In [3]:
# Configurăm providerii și cheile API din fișierul .env

load_dotenv()

BASE_URLS = {
    "gemini": "https://generativelanguage.googleapis.com/v1beta/openai/",
    "openrouter": "https://openrouter.ai/api/v1"
}

API_KEYS = {
    "gemini": os.getenv("GEMINI_API_KEY"),
    "openrouter": os.getenv("OPENROUTER_API_KEY")
}

def make_client(provider):
    """Creează clientul API pentru providerul ales."""
    return OpenAI(
        api_key=API_KEYS[provider],
        base_url=BASE_URLS[provider]
    )

## 2. Funcție helper — trimitem același prompt la orice model

În loc să scriem același cod de 3 ori, facem o funcție.

In [4]:
# varianta minimala

# fara functie
client = make_client("gemini")
prompt = "Explică în 2 propoziții ce este un LLM."
response = client.chat.completions.create(
    model="gemini-2.5-flash-lite",
    messages=[
        {"role": "user", "content": prompt}
    ]
)
print(response.choices[0].message.content)

# cu functie
def ask(provider, model, prompt):
    client = make_client(provider)

    messages = [
        {"role": "user", "content": prompt}
    ]
    response = client.chat.completions.create(
        model=model,
        messages=messages
    )
    return response.choices[0].message.content

# iar functia poate fi apelata astfel:
raspuns = ask(
    provider="gemini",
    model="gemini-2.5-flash-lite",
    prompt="Explică în 2 propoziții ce este un LLM."
)

print(raspuns)

Un LLM (Large Language Model) este un model de inteligență artificială antrenat pe cantități masive de text și cod, capabil să înțeleagă, să genereze și să manipuleze limbajul uman cu o fluență remarcabilă. Aceste modele pot efectua o gamă largă de sarcini lingvistice, cum ar fi traducerea, rezumarea, scrierea creativă și răspunsul la întrebări, prin identificarea și utilizarea modelelor complexe din datele pe care au fost antrenate.
Un LLM (Large Language Model) este un tip de model de inteligență artificială antrenat pe cantități uriașe de text pentru a înțelege, genera și manipula limbajul uman. Această capacitate îi permite să răspundă la întrebări, să scrie texte creative, să traducă limbi și să rezume informații.


In [5]:
from openai import RateLimitError, APIError, AuthenticationError
import json

def ask(provider, model, prompt, system=None, temperature=0.7, json_schema=None):
    """Trimite un prompt la model. Poate returna text simplu sau JSON structurat."""

    client = make_client(provider)

    messages = []

    if system:
        messages.append({"role": "system", "content": system})

    messages.append({"role": "user", "content": prompt})

    extra_args = {}

    if json_schema:
        extra_args["response_format"] = {
            "type": "json_schema",
            "json_schema": json_schema
        }

    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature,
            **extra_args
        )

        text = response.choices[0].message.content.strip()

        if json_schema:
            return json.loads(text)

        return text

    except RateLimitError:
        return f"[Eroare: quota/rate limit pentru modelul {model}.]"

    except AuthenticationError:
        return "[Eroare: API key invalidă sau lipsă. Verifică .env.]"

    except APIError as e:
        return f"[Eroare API: {e}]"

    except Exception as e:
        return f"[Eroare: {type(e).__name__} — {e}]"

## 3. Test 1 — Calitatea pe limba română

Testăm dacă modelele înțeleg și răspund corect în română.

In [6]:
PROMPT_RO = """
Rezumă următoarea situație politică în 3 puncte clare și obiective:
"Coaliția de guvernare din România discută despre comasarea alegerilor parlamentare cu cele prezidențiale. 
Opoziția critică dur această propunere, susținând că este un atac la democrație, în timp ce puterea 
afirmă că măsura ar reduce costurile și ar crește prezența la vot.
"""

for provider, model_id, nume in MODELE:
    print("\n---", nume, "---")

    raspuns = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT_RO,
        temperature=0.2
    )

    print(raspuns)


--- Gemini 2.5 Flash Lite ---
Iată un rezumat al situației politice în 3 puncte clare și obiective:

1.  **Propunerea de comasare a alegerilor:** Coaliția de guvernare din România analizează posibilitatea organizării alegerilor parlamentare și prezidențiale în același timp.
2.  **Argumentele puterii:** Guvernarea susține că această comasare ar genera economii financiare și ar stimula participarea cetățenilor la vot.
3.  **Criticile opoziției:** Opoziția consideră propunerea o amenințare la adresa principiilor democratice.

--- Gemini 2.5 Flash ---
Iată rezumatul în 3 puncte clare și obiective:

1.  **Propunere:** Coaliția de guvernare din România discută comasarea alegerilor parlamentare cu cele prezidențiale.
2.  **Argumente pro:** Puterea justifică această măsură prin reducerea costurilor electorale și creșterea prezenței la vot.
3.  **Argumente contra:** Opoziția critică vehement propunerea, considerând-o un atac la democrație.

--- OpenRouter Free ---
1. **Coaliția de guvernare pr

## 4. Test 2 — Urmează instrucțiunile din system prompt+ adnotare

Vedem dacă modelele respectă rolul dat prin `system`.

In [7]:
SYSTEM = """
Ești un expert în sociologie politică care identifică tipologia discursului.
Răspunzi scurt, clar și nu inventezi informații.
"""

PROMPT = """
Analizează următorul comentariu politic:
"Iarăși promisiuni înainte de vot. După ce trec alegerile, uită de noi toți și își văd de afacerile lor."

Răspunde în 4 linii:
Ton:
Emoție dominantă:
Țintă principală:
Populism: da/nu
"""

for provider, model, name in MODELE:
    print("\n---", name, "---")
    print(ask(
        provider=provider,
        model=model,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0
    ))


--- Gemini 2.5 Flash Lite ---
Ton: Critic, cinic.
Emoție dominantă: Dezamăgire, neîncredere.
Țintă principală: Clasa politică, politicienii.
Populism: Da.

--- Gemini 2.5 Flash ---
Ton: Critic, cinic, acuzator.
Emoție dominantă: Frustrare, neîncredere.
Țintă principală: Clasa politică (în general).
Populism: Da.

--- OpenRouter Free ---
Ton: Cinic și acuzator.  
Emoție dominantă: Dezamăgire și neîncredere.  
Țintă principală: Politicienii/clasa politică.  
Populism: da


## 5. Test 3 — Output structurat (JSON)

Agenții noștri vor trebui să returneze date structurate.
Testăm dacă modelele pot produce JSON valid la cerere.

In [8]:
SCHEMA_ADNOTARE = {
    "name": "adnotare_comentariu_politic",
    "schema": {
        "type": "object",
        "properties": {
            "ton": {
                "type": "string",
                "enum": ["pozitiv", "negativ", "neutru"]
            },
            "emotie_dominanta": {
                "type": "string",
                "enum": ["furie", "frica", "speranta", "dezamagire", "ironie", "neutru"]
            },
            "tinta_principala": {
                "type": "string"
            },
            "populism": {
                "type": "boolean"
            },
            "explicatie_scurta": {
                "type": "string"
            }
        },
        "required": [
            "ton",
            "emotie_dominanta",
            "tinta_principala",
            "populism",
            "explicatie_scurta"
        ],
        "additionalProperties": False
    }
}

In [9]:
COMENTARIU = "Toți politicienii fură, iar oamenii simpli plătesc nota. Nimeni nu mai ascultă poporul."

SYSTEM = "Ești un asistent de cercetare care adnotează comentarii politice."

PROMPT = f"Analizează riguros acest comentariu și extrage datele conform schemei JSON: {COMENTARIU}"

for provider, model_id, nume in MODELE:
    print("\n---", nume, "---")

    rezultat = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0.1,
        json_schema=SCHEMA_ADNOTARE
    )

    print(rezultat)


--- Gemini 2.5 Flash Lite ---
{'ton': 'negativ', 'emotie_dominanta': 'furie', 'tinta_principala': 'politicienii', 'populism': True, 'explicatie_scurta': "Comentariul exprimă frustrare și furie față de politicieni, acuzându-i de corupție și ignorarea voinței poporului. Se folosește o dihotomie simplistă între 'politicieni' și 'oameni simpli', specifică discursului populist."}

--- Gemini 2.5 Flash ---
{'ton': 'negativ', 'emotie_dominanta': 'furie', 'tinta_principala': 'politicienii', 'populism': True, 'explicatie_scurta': 'Comentariul exprimă furie și dezamăgire față de toți politicienii, acuzându-i de corupție și de ignorarea poporului, în timp ce oamenii simpli suferă consecințele.'}

--- OpenRouter Free ---
{'emotie_dominanta': 'dezamagire', 'explicatie_scurta': 'Comentariul este general și negativ, subliniazând diferențele între politicieni și oamenii comune, iar subiectul este critic de la o perspectivă de care nu consideră opinia publicului.', 'populism': True, 'tinta_principala'

## 6. Test 4 — Stabilitate la temperature diferite

Un model bun pentru agenți trebuie să fie **stabil** — același input, răspunsuri similare.
Testăm cu Gemini (poți schimba cu orice model).

In [10]:
PROMPT_STAB = """
Explică în exact două propoziții de ce este importantă independența justiției într-o democrație. 
Folosește un ton academic.
"""

TEMPERATURI = [0.1, 0.7, 1.2]

print("[ Test 4 — stabilitate: același prompt, temperaturi diferite ]")

for provider, model_id, nume in MODELE:
    print("\n" + "=" * 60)
    print(f"[ {nume} ]")

    for temp in TEMPERATURI:
        raspuns = ask(
            provider=provider,
            model=model_id,
            prompt=PROMPT_STAB,
            temperature=temp
        )

        print(f"\ntemperature={temp}:")
        print(raspuns)

[ Test 4 — stabilitate: același prompt, temperaturi diferite ]

[ Gemini 2.5 Flash Lite ]

temperature=0.1:
Independența justiției este fundamentală într-o democrație, deoarece asigură aplicarea imparțială a legii și protecția drepturilor cetățenilor împotriva abuzurilor de putere. Aceasta permite judecătorilor să ia decizii bazate exclusiv pe fapte și pe lege, fără influențe politice sau economice, consolidând astfel statul de drept și încrederea publică în instituțiile democratice.

temperature=0.7:
Independența justiției este fundamentală într-o democrație, deoarece asigură aplicarea imparțială a legii și protecția drepturilor cetățenilor împotriva potențialelor abuzuri ale puterii executive sau legislative. Această independență permite judecătorilor să ia decizii bazate exclusiv pe fapte și lege, fără influențe externe, consolidând astfel statul de drept și încrederea publică în sistemul judiciar.

temperature=1.2:
Independența justiției este esențială într-o democrație, deoarece p

## 7. Alegerea modelului pentru proiect

Completați tabelul după testele de mai sus. Nu căutați „cel mai bun model” în general, ci modelul cel mai potrivit pentru proiectul vostru.
| Model | Răspunde bine în română? | Respectă instrucțiunile? | Merge pentru adnotare? | Are erori / quota? | Observație scurtă |
|---|---|---|---|---|---|
| Gemini 2.5 Flash Lite | da | da | da  | nu | |
| OpenRouter Free | da | parțial | parțial | nu | |
| Llama / alt model testat | da / nu / parțial | da / nu / parțial | da / nu / parțial | da / nu | |
### Decizie
**Model principal ales:**  Gemini 2.5 Flash
**Model de rezervă:**  OpenRouter
**Temperature recomandată:**  0.7
**De ce am ales acest model?**  Gemini a fost mult mai stabil la formatul JSON. La temperatura 1.2, modelele din OpenRouter au început să dea răspunsuri prea lungi și mai puțin relevante față de prompt-ul inițial.
Scrieți 2-3 propoziții. Menționați calitatea răspunsului, stabilitatea și dacă modelul poate fi folosit pentru adnotarea comentariilor.

## 8. Configurația finală a proiectului

putem să copiem asta in core/config.py

In [17]:
# core/config.py
# Configurația modelului ales de echipă după testele din Cursul 2.
# Nu puneți chei API aici. Cheile rămân doar în fișierul local .env.
PROVIDER_PRINCIPAL = "gemini"
MODEL_PRINCIPAL = "gemini-2.5-flash-lite"
PROVIDER_FALLBACK = "openrouter"
MODEL_FALLBACK = "openrouter/free"
TEMPERATURE = 0.2

---

## Livrabile C2

Până la cursul următor:

- [ ] Notebook completat cu 2-3 modele testate
- [ ] Matricea de decizie completată cu observații reale
- [ ] README actualizat cu modelul ales și justificarea
- [ ] `.env` configurat cu cheia pentru modelul ales